# 05 — Three conditional flows for the test-statistic distribution

We now freeze the response network, the inference networks and the misspecified likelihood from notebooks 02–04. This notebook learns the distribution of the resulting statistic. It does not improve the event model or solve profiling again.

We build three scalar, conditional normalizing flows:

1. $Q_\phi$ describes response-matched **reference** experiments.
2. $K_\eta$ corrects the residual reference-distribution error on another set of experiments.
3. $G_\psi$ converts that reference CDF into the **physical simulator** CDF.

The final distribution is $F_\mu(t)=G_\psi(K_\eta(Q_\phi(t\mid\mu)\mid\mu)\mid\mu)$. Each flow has an explicit inverse, so critical values require no integration or root finding. Calibration quality still has to be measured on independent experiments.

## Setup and shared run directory

Run this cell first. Use the same `RUN_NAME` in all five notebooks. Colab can keep the run on Drive; locally, open Jupyter in this directory. Use a fresh kernel when switching from the older exercises. Training and samples remain in the run directory.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

TAG = "sb10_exppoly_v2_20260930"  # New tag: rebuild all five stages.
RUN_NAME = TAG
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("TAG:", TAG)
print("Run:", RUN)
if "CALIBRATION_RUN" in os.environ:
    print("CALIBRATION_RUN overrides the tagged path; use a fresh directory for this model.")
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from model import load_config, with_model_epsilon, fit
from sampling import sample_experiment
from utils import HybridModel
from inference import load_inference
from statistic_flows import (train_flow, ComposedCDF, generate_statistics,
                             flow_identity, coverage_summary)

config = load_config(RUN)
epsilon = json.loads((RUN / "misspecification.json").read_text())["epsilon"]
device = "cuda" if torch.cuda.is_available() else "cpu"
hybrid = HybridModel.load(RUN, device=device)
inference = load_inference(RUN, device=device)
inference_metadata = json.loads((RUN / "inference/metadata.json").read_text())
assert inference_metadata["hybrid_model_id"] == hybrid.model_id, "Rebuild notebook 04 with this hybrid model."
assert inference_metadata["epsilon"] == epsilon
config = with_model_epsilon(config, epsilon)
assert inference_metadata["config"] == config
FLOW_DIR = RUN / "statistic_flows"
FLOW_DIR.mkdir(exist_ok=True)
identity = flow_identity(RUN, {**config, "epsilon": epsilon})
print("Frozen inference / hybrid identity:", identity[:16])
print("Response at physical endpoints:", inference.response(np.array(config["mu_range"])))

## 1. Define the experiment and statistic before calibrating them

At physical truth $\mu$ the response network supplies $(g(\mu),a(\mu))$. Reference toys are generated from the bad hybrid model at $(\nu,\alpha)=(g(\mu),a(\mu))$; simulator toys come from the physical signal/background model at $(\mu,\alpha_{\rm gen}=0)$. Both are tested at $\nu=g(\mu)$ using the same frozen inference networks.

In both ensembles, $y_{\rm aux}\sim\mathrm{Uniform}(-2,2)$, as specified for this study. The likelihood still contains the Gaussian term $-(y_{\rm aux}-\alpha)^2/2$. These are two different choices: the likelihood constraint does not turn the toy ensemble into a Gaussian-auxiliary ensemble. Our coverage statements apply only to the uniform-auxiliary ensemble with $\alpha_{\rm gen}=0$.

The raw amortized likelihood difference is
$$\widetilde T_\mu(D)=2\{\mathrm{NLL}(g(\mu),\widehat\alpha_D(g(\mu)))-\mathrm{NLL}(\widehat\nu_D,\widehat\alpha_D)\}.$$
Because both optimizers are approximate, this can be negative. We retain and report that signed gap, and explicitly define the statistic calibrated here as
$$T_\mu^{\rm NN}(D)=\max(0,\widetilde T_\mu(D)).$$
This definition is not a claim that profiling errors have disappeared. A large negative tail is a reason to improve notebook 04 before using its inference. Independent direct fits below quantify the difference. Calibration can give the defined NN statistic the right null distribution without making it equal to the exact profile-likelihood statistic.

In [ ]:
# Increase these numbers after the first closure check, without reusing old
# toy rows in a new holdout. Defaults are a first calibration study, not a
# promised precision target.
N_BASE = 12_000
N_REFERENCE_RESIDUAL = 6_000
N_SIMULATOR_RESIDUAL = 12_000
N_HOLDOUT = 2_000  # per source
N_REFERENCE_AUDIT = 1_000_000  # fresh proposal bank for reference holdout
N_REFERENCE_LARGER = 2_000_000 # independent larger-bank diagnostic
N_BANK_COMPARISON = 500
FLOW_EPOCHS = 150
FLOW_BINS = 64
SEED = int(config["seed"]) + 5_000


def design_mu(n, seed):
    rng = np.random.default_rng(seed)
    mu = rng.uniform(*config["mu_range"], n)
    # Give exact boundaries data: a continuous design never samples them.
    edge_n = n // 10
    mu[:edge_n] = config["mu_range"][0]
    mu[edge_n:2 * edge_n] = config["mu_range"][1]
    rng.shuffle(mu)
    return mu


def ensemble(name, source, n, seed, sampler=None, bank_id="integration"):
    return generate_statistics(
        inference, hybrid, config, design_mu(n, seed), source=source,
        seed=seed + 100, epsilon=epsilon,
        checkpoint=FLOW_DIR / f"{name}.npz", identity=identity,
        reference_sampler=sampler, reference_bank=bank_id,
    )

base_toys = ensemble("reference_base", "reference", N_BASE, SEED)
reference_toys = ensemble("reference_residual", "reference", N_REFERENCE_RESIDUAL, SEED + 1)
simulator_toys = ensemble("simulator_residual", "simulator", N_SIMULATOR_RESIDUAL, SEED + 2)

Reference toy generation uses cached ratio features on the five-million-event reference integration bank from notebook 02. Sampling uses cached polynomial cumulative sums for the normalized exp-poly densities, so ratios and full-bank weights are not re-evaluated for every toy. The signal mixture is applied after normalizing the two processes, exactly as in the frozen likelihood. This is a weighted empirical approximation to the continuous hybrid distribution. Toy draws and random-number streams are disjoint; the shared bank still leaves a common integration approximation. We therefore audit the reference distribution with a fresh independent bank later and compare a larger-bank result. The physical simulator holdout consists of fresh continuous events, not reference-bank events. The saved cache records the frozen model identity; changing any network or the mixture strength requires new calibration toys.

In [ ]:
def statistic(toys):
    # This explicit definition creates an atom at zero when a signed NN gap
    # is negative. Raw gaps remain saved for diagnostics.
    return np.maximum(toys["raw_statistic"], 0.)


def gap_summary(name, toys):
    raw = toys["raw_statistic"]
    return {"ensemble": name, "n": len(raw),
            "negative fraction": np.mean(raw < 0),
            "minimum raw gap": raw.min(),
            "1% raw gap": np.quantile(raw, .01),
            "zero statistic fraction": np.mean(statistic(toys) == 0)}

pd.DataFrame([gap_summary(name, toys) for name, toys in [
    ("Q reference", base_toys), ("K reference", reference_toys),
    ("G simulator", simulator_toys)]])

## 2. A simple scalar flow that respects the zero atom

A continuous flow alone cannot describe a point mass. An exact statistic can have a physical atom at a parameter boundary; our approximate statistic can also have an **optimization-induced** atom from the explicit $\max(0,\widetilde T)$ definition. We model the total observed atom and keep these explanations separate.

For $t>0$, the fixed map $h(t)=\sqrt{t}/(\sqrt{t}+\sqrt{s})$ transforms the positive half-line into $(0,1)$; here $s=1$. A small NN takes $\mu$ and predicts positive interval probabilities. Their cumulative sums define a monotone linear-spline CDF on $(0,1)$, with an analytic inverse. Two endpoint indicators in the conditioning input let the model represent boundary-specific behavior. For $Q$, the same NN also predicts the zero probability $p_0(\mu)$:
$$Q_\phi(t\mid\mu)=p_0(\mu)+(1-p_0(\mu))C_\phi(h(t)\mid\mu),\quad t>0,$$
with $Q_\phi(0\mid\mu)=p_0(\mu)$. Training maximizes the mixed likelihood: $\log p_0$ for zeros and the continuous log density, including the Jacobian, for positive observations.

This is a deliberately small one-dimensional normalizing flow. The word “linear” describes its CDF between spline knots; it does not impose a quadratic likelihood or a chi-square distribution. The finite number of knots is a density-model approximation checked by the holdout plots.

In [ ]:
Q, history_Q = train_flow(
    base_toys["mu"], statistic(base_toys), statistic=True,
    mu_range=config["mu_range"], bins=FLOW_BINS, epochs=FLOW_EPOCHS,
    seed=SEED + 10, device=device, checkpoint=FLOW_DIR / "Q_reference.pt",
)
Q_only = ComposedCDF(Q)

## 3. Correct reference error with independent reference toys

The probability-integral transform must also respect an atom. We use
$$U=F(T^-\mid\mu)+V\{F(T\mid\mu)-F(T^-\mid\mu)\},\qquad V\sim\mathrm{Uniform}(0,1).$$
For a positive continuous observation, this is just $F(T\mid\mu)$. For a zero it spreads the observation uniformly across its CDF jump. The residual flow $K_\eta$ learns the distribution of these $Q$-PIT values on the second reference sample. If $Q$ were exact, $K$ would be the identity.

The improved reference CDF is $H_\mu(t)=K_\eta(Q_\phi(t\mid\mu)\mid\mu)$. At zero its mass is $H_\mu(0)$; it is not discarded by the residual correction.

In [ ]:
u_reference = Q_only.pit(reference_toys["mu"], statistic(reference_toys),
                         np.random.default_rng(SEED + 11))
K, history_K = train_flow(
    reference_toys["mu"], u_reference, mu_range=config["mu_range"],
    bins=FLOW_BINS, epochs=FLOW_EPOCHS, seed=SEED + 12, device=device,
    checkpoint=FLOW_DIR / "K_reference_residual.pt",
)
H = ComposedCDF(Q, K)

## 4. Correct the remaining simulator/reference difference

We evaluate $H$ on independent physical simulator experiments and train $G_\psi$ on their randomized PIT values. This residual captures both the intentional mixture misspecification and whatever reference/statistic-density error remains. It therefore depends on the frozen response and inference networks.

For zeros, randomization is performed **again using the current CDF interval** $[0,H_\mu(0)]$. Applying $K$ to a previously randomized value would not in general produce a uniform randomization over the new atom interval.

The final CDF and generalized inverse are
$$F_\mu(t)=G_\psi(H_\mu(t)\mid\mu),\qquad
F_\mu^{-1}(p)=Q_\phi^{-1}\bigl(K_\eta^{-1}(G_\psi^{-1}(p\mid\mu)\mid\mu)\mid\mu\bigr).$$
If $p\leq F_\mu(0)$, the quantile is exactly zero. No logarithmic offset, discarded zeros or continuous approximation to their mass is needed.

In [ ]:
u_simulator = H.pit(simulator_toys["mu"], statistic(simulator_toys),
                    np.random.default_rng(SEED + 13))
G, history_G = train_flow(
    simulator_toys["mu"], u_simulator, mu_range=config["mu_range"],
    bins=FLOW_BINS, epochs=FLOW_EPOCHS, seed=SEED + 14, device=device,
    checkpoint=FLOW_DIR / "G_simulator_residual.pt",
)
F = ComposedCDF(Q, K, G)
(FLOW_DIR / "manifest.json").write_text(json.dumps({
    "frozen_identity": identity, "epsilon": epsilon,
    "statistic": "max(0, raw amortized likelihood gap)",
    "auxiliary_law": "Uniform(-2,2)", "simulator_alpha": 0.,
    "reference_generation": "bad hybrid at response(mu)",
    "mu_range": config["mu_range"], "seed": SEED,
    "training_counts": [N_BASE, N_REFERENCE_RESIDUAL, N_SIMULATOR_RESIDUAL],
}, indent=2))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, name, history in zip(axes, ["Q reference", "K reference residual", "G simulator residual"],
                             [history_Q, history_K, history_G]):
    ax.plot(history[:, 0], label="training")
    ax.plot(history[:, 1], label="validation")
    ax.set(xlabel="epoch", ylabel="negative log likelihood", title=name)
    ax.legend()
fig.tight_layout()
fig.savefig(FLOW_DIR / "flow_training.png", dpi=150)

## 5. Inversion and critical values

For mixed distributions the usual equality $F(F^{-1}(p))=p$ changes inside an atom: the correct relation is $F(t^-)\leq p\leq F(t)$ at $t=F^{-1}(p)$. Outside the zero atom, the CDF and inverse should agree to numerical precision. This checks the implementation, not statistical coverage.

The nonrandomized rule $T_\mu^{\rm NN}\leq F_\mu^{-1}(1-\beta)$ can overcover if a critical quantile lies inside the zero atom. A randomized PIT rule gives exact nominal probabilities for an exact CDF; we report its uniformity as a diagnostic, while retaining the conservative, nonrandomized critical-value rule for inference.

In [ ]:
mu_grid = np.linspace(*config["mu_range"], 101)
levels = np.array([.50, .68, .90, .95, .99])
quantiles = F.ppf(mu_grid[:, None], levels[None, :])
left = F.cdf(mu_grid[:, None], quantiles, left=True)
right = F.cdf(mu_grid[:, None], quantiles)
quantile_error = np.maximum(left - levels, levels - right)
print("Maximum generalized-inverse inequality violation:", quantile_error.max())
assert quantile_error.max() < 1e-9
positive = quantiles > 0
print("Positive-quantile inverse error:", np.max(np.abs(right - levels)[positive]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for level in [.68, .90, .95, .99]:
    axes[0].plot(mu_grid, H.ppf(mu_grid, level), ls="--", alpha=.7)
    axes[0].plot(mu_grid, F.ppf(mu_grid, level), label=f"{level:.0%} calibrated")
axes[0].set(xlabel=r"physical $\mu$", ylabel=r"critical $T^{NN}$",
            title="Solid: calibrated; dashed: reference")
axes[0].legend()
axes[1].plot(mu_grid, Q.cdf(mu_grid, 0.), label="Q zero mass")
axes[1].plot(mu_grid, H.cdf(mu_grid, 0.), label="reference zero mass")
axes[1].plot(mu_grid, F.cdf(mu_grid, 0.), label="simulator zero mass")
axes[1].set(xlabel=r"physical $\mu$", ylabel=r"$P(T^{NN}=0)$")
axes[1].legend()
fig.tight_layout()
fig.savefig(FLOW_DIR / "critical_curves.png", dpi=150)
np.savez_compressed(FLOW_DIR / "critical_values.npz", mu=mu_grid,
                    levels=levels, calibrated=quantiles,
                    reference=H.ppf(mu_grid[:, None], levels[None, :]))

## 6. Independent held-out experiments

These samples were not used for the response network, profiling networks, any of the three flow fits, or their validation losses. We check conditional PIT distributions, nonrandomized coverage and the predicted zero probability. Separate boundary checks matter: global plots can hide a failure at $\mu=0$.

Monte Carlo error bars describe the finite holdout, not training uncertainty. The default holdout is meant to find visible failures; demonstrating sub-percent conditional coverage would require substantially more toys per physical-parameter region.

In [ ]:
# A cache hit needs no proposal generation. A new holdout uses fresh reference
# events independent of the shared bank used for Q and K training.
audit_sampler = (None if (FLOW_DIR / "reference_holdout.npz").exists() else
                 hybrid.make_sampler(N_REFERENCE_AUDIT, SEED + 120))
reference_holdout = ensemble("reference_holdout", "reference", N_HOLDOUT, SEED + 20,
                             sampler=audit_sampler,
                             bank_id=f"fresh_{N_REFERENCE_AUDIT}_{SEED + 120}")
del audit_sampler
simulator_holdout = ensemble("simulator_holdout", "simulator", N_HOLDOUT, SEED + 21)
display(pd.DataFrame([gap_summary("reference holdout", reference_holdout),
                     gap_summary("simulator holdout", simulator_holdout)]))

u_hold_reference = H.pit(reference_holdout["mu"], statistic(reference_holdout),
                         np.random.default_rng(SEED + 22))
u_hold_simulator_before = H.pit(simulator_holdout["mu"], statistic(simulator_holdout),
                               np.random.default_rng(SEED + 23))
u_hold_simulator = F.pit(simulator_holdout["mu"], statistic(simulator_holdout),
                         np.random.default_rng(SEED + 24))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for u, label in [(u_hold_reference, "reference after K"),
                 (u_hold_simulator_before, "simulator before G"),
                 (u_hold_simulator, "simulator after G")]:
    axes[0].hist(u, bins=20, range=(0, 1), density=True, histtype="step", label=label)
axes[0].axhline(1, color="k", ls=":")
axes[0].set(xlabel="randomized PIT", ylabel="density")
axes[0].legend()
for low, high in zip(np.linspace(*config["mu_range"], 5)[:-1],
                     np.linspace(*config["mu_range"], 5)[1:]):
    mask = ((simulator_holdout["mu"] > low) & (simulator_holdout["mu"] < high))
    values = np.sort(u_hold_simulator[mask])
    axes[1].plot(values, np.arange(1, len(values) + 1) / len(values) - values,
                 label=rf"${low:.2f}<\mu<{high:.2f}$")
axes[1].axhline(0, color="k", ls=":")
axes[1].set(xlabel="PIT", ylabel="empirical CDF minus uniform CDF")
axes[1].legend()
fig.tight_layout()
fig.savefig(FLOW_DIR / "heldout_pit.png", dpi=150)

In [ ]:
coverage = pd.DataFrame(coverage_summary(simulator_holdout["mu"],
                                         statistic(simulator_holdout), F))
display(coverage)
coverage.to_csv(FLOW_DIR / "heldout_coverage.csv", index=False)

atom_rows = []
for source, toys, cdf in [("reference", reference_holdout, H),
                           ("simulator", simulator_holdout, F)]:
    for endpoint in config["mu_range"]:
        mask = toys["mu"] == endpoint
        observed = np.mean(statistic(toys)[mask] == 0)
        endpoint_critical = float(cdf.ppf(endpoint, .95))
        coverage_95 = np.mean(statistic(toys)[mask] <= endpoint_critical)
        atom_rows.append({"source": source, "mu": endpoint, "n": int(mask.sum()),
                          "response g(mu)": float(inference.response(np.array([endpoint]))[0, 0]),
                          "observed atom": observed,
                          "predicted atom": float(cdf.cdf(endpoint, 0.)),
                          "95% critical": endpoint_critical, "95% coverage": coverage_95,
                          "coverage SE": np.sqrt(coverage_95 * (1 - coverage_95) / mask.sum()),
                          "binomial SE": np.sqrt(observed * (1 - observed) / mask.sum())})
display(pd.DataFrame(atom_rows))

fig, ax = plt.subplots(figsize=(8, 4))
for level in [.68, .90, .95]:
    rows = coverage[coverage.level == level]
    centers = .5 * (rows.mu_low + rows.mu_high)
    ax.errorbar(centers, rows.coverage - level, yerr=rows.standard_error,
                marker="o", label=f"{level:.0%}")
ax.axhline(0, color="k", ls=":")
ax.set(xlabel=r"physical $\mu$", ylabel="coverage minus nominal",
       title="Physical simulator, uniform auxiliary, one-sigma toy errors")
ax.legend()
fig.tight_layout()
fig.savefig(FLOW_DIR / "heldout_coverage.png", dpi=150)

The fresh reference bank is itself finite. The next check repeats reference sampling with a larger independent bank and compares the mean PIT and 95% acceptance rate. The proposal effective sample size is also reported. Differences comparable to toy standard errors do not prove exact reference generation, but a visible shift identifies a bank-size limitation before attributing it to flow training. The two banks use independent seeds; their Monte Carlo fluctuations are not paired.

In [ ]:
larger_sampler = (None if (FLOW_DIR / "reference_larger_bank.npz").exists() else
                  hybrid.make_sampler(N_REFERENCE_LARGER, SEED + 121))
reference_larger = ensemble("reference_larger_bank", "reference", N_BANK_COMPARISON,
                            SEED + 25, sampler=larger_sampler,
                            bank_id=f"fresh_{N_REFERENCE_LARGER}_{SEED + 121}")
del larger_sampler
bank_rows = []
for label, toys in [("independent audit bank", reference_holdout),
                     ("independent larger bank", reference_larger)]:
    pit = H.pit(toys["mu"], statistic(toys), np.random.default_rng(SEED + 26))
    accept = statistic(toys) <= H.ppf(toys["mu"], .95)
    bank_rows.append({"bank": label, "n toys": len(pit), "mean PIT": pit.mean(),
                      "PIT mean SE": pit.std(ddof=1) / np.sqrt(len(pit)),
                      "coverage 95%": accept.mean(),
                      "coverage SE": np.sqrt(accept.mean() * (1 - accept.mean()) / len(pit)),
                      "median proposal ESS": np.median(toys["proposal_ess"])})
display(pd.DataFrame(bank_rows))

## 7. Check what is being calibrated against direct profiling

Finally, compare network and direct-fit statistics on fresh unbinned physical experiments. The critical values above belong to $T^{\rm NN}$: substituting a direct-fit statistic into the same acceptance rule is only a diagnostic and is not a calibrated procedure.

Inspect the statistic differences, conditional nuisance fits and negative raw gaps together. Good PIT closure with large direct-fit discrepancies means calibration has corrected the distribution of an inaccurate statistic. It does not show that the optimizer networks have learned profiling well. In particular, the zero-mass table cannot by itself distinguish a genuine boundary atom from optimization error.

In [ ]:
N_DIRECT = 30
rng = np.random.default_rng(SEED + 30)
direct_rows = []
for mu in np.linspace(*config["mu_range"], N_DIRECT):
    toy = sample_experiment(float(mu), 0., config, rng)
    anchors = hybrid.anchors(toy["x"])
    nu, _ = inference.response(np.array([mu]))[0]
    nn_result = inference.evaluate(anchors, toy["auxiliary"], float(nu))
    global_fit = fit(anchors, toy["auxiliary"], config)
    conditional_fit = fit(anchors, toy["auxiliary"], config, fixed_nu=float(nu))
    direct_t = 2 * (conditional_fit["nll"] - global_fit["nll"])
    raw_t = float(np.asarray(nn_result["statistic"]))
    direct_rows.append({"mu": mu, "raw NN statistic": raw_t,
                        "NN statistic": max(0., raw_t), "direct statistic": direct_t,
                        "conditional alpha NN": float(np.asarray(nn_result["conditional_alpha"])),
                        "conditional alpha direct": conditional_fit["alpha"],
                        "direct success": global_fit["success"] and conditional_fit["success"]})
direct = pd.DataFrame(direct_rows)
display(direct)
print("Median / 90% absolute statistic difference:",
      np.quantile(np.abs(direct["NN statistic"] - direct["direct statistic"]), [.5, .9]))
direct.to_csv(FLOW_DIR / "direct_fit_comparison.csv", index=False)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(direct["direct statistic"], direct["NN statistic"], c=direct.mu)
limit = max(direct["direct statistic"].max(), direct["NN statistic"].max())
ax.plot([0, limit], [0, limit], "k--")
ax.set(xlabel="direct unbinned statistic", ylabel="amortized statistic")
fig.tight_layout()
fig.savefig(FLOW_DIR / "direct_fit_comparison.png", dpi=150)

The saved result is a calibrated distribution for one explicitly defined experiment ensemble and one frozen approximate statistic. Before applying it more broadly, we need closure for the intended nuisance-generation and auxiliary-observation laws. Extending the conditioner from $\mu$ to $(\mu,\alpha_{\rm gen})$ is possible, but uniform coverage over unknown nuisance parameters is a further inference problem; it is not established by the nominal-$\alpha$ validation here.